In [1]:
import pandas as pd
import numpy as np 
import seaborn as sns 
import plotly.express as px 
from pathlib import Path
import geopandas as gps

In [ ]:
df = pd.read_excel("dados.xlsx", sheet_name = "Dados")

df

In [39]:
teste = df.groupby(["data","setor","turno"], as_index = False).aggregate(peso = ("producao", "sum"), dia_semana = ("dia_semana", "unique"))
teste["dia_semana"] = [i[0] for i in teste["dia_semana"]]
teste["data"] = pd.to_datetime(teste["data"], dayfirst= True)
teste = teste[(teste["data"]>="20/05/2025") & (teste["data"]<="20/06/2025") & (teste["dia_semana"].isin(["seg","qua","sex"]))]
px.line(teste, x = "data", y = "peso", color = "setor")

In [74]:
def serie_historica(dados, data_ini, data_fin, dia_sem ,turn ,var):
    teste = dados
    teste["data"] = pd.to_datetime(teste["data"], dayfirst= True)
    teste = dados.groupby(["data", "setor", "turno"], as_index = False).aggregate(var = (var,"sum"), dia_semana = ("dia_semana", "unique"))
    teste["dia_semana"] = [i[0] for i in teste["dia_semana"]]
    teste = teste[(teste["data"]>=data_ini) & (teste["data"]<=data_fin) & (teste["turno"].isin(turn)) & (teste["dia_semana"].isin(dia_sem))]
    
    teste.sort_values(by = "data", inplace = True)
    teste["indexes"] = teste["data"].dt.strftime("%d/%m/%y") + "<br>" + teste["dia_semana"]
    
    figura = px.line(teste, x = "indexes", y = "var", color = "setor")
    figura.show()
    
    return teste

In [75]:
treino  = serie_historica(df, "20/05/2025", "20/10/2025", ["seg","qua","sex"], ["DIURNO"] ,"horas_coleta")

In [89]:
def medias(dados, data_ini, data_fin, dia_sem ,turn ,var):
    teste = dados
    teste["data"] = pd.to_datetime(teste["data"], dayfirst= True)
    
    teste = teste[(teste["data"] >= data_ini) & (teste["data"] <= data_fin) & (teste["turno"].isin(turn)) & (teste["dia_semana"].isin(dia_sem))].copy()
    
    teste = teste.groupby(["setor", "dia_semana" ,"turno"], as_index= False)[var].mean()
    
    figura = px.bar(teste, x = var, y = "setor", color = "dia_semana", orientation= "h")
    figura.update_layout(barmode = "group")
    figura.show()
    

In [90]:
medias(df, "20/05/2025", "20/06/2025",["seg","qua","sex"], ["DIURNO"], "producao")

In [ ]:
!pip install geopandas

In [103]:

mapa_atual = gps.read_file("Atual/01.shp")

for i in range(2,22):
    if i <10: 
        mapa = gps.read_file(f"Atual/0{i}.shp")
        mapa_atual = pd.concat([mapa_atual, mapa])
    else:
        mapa = gps.read_file(f"Atual/{i}.shp")
        mapa_atual = pd.concat([mapa_atual, mapa])

In [105]:
mapa_atual = mapa_atual.to_crs(epsg=4326)

mapa_atual.to_file("mapa_atual.geojson", driver = "GeoJSON")

In [13]:
mapa_atual = gps.read_file("mapa_atual.geojson")
df = pd.read_excel("dados.xlsx",sheet_name = "Dados")
teste = df.groupby( "setor", as_index = False)["producao"].mean()
teste.drop([21, 22], axis = 0, inplace = True)
teste["setor"] = [f"0{i}" if i < 10 else f"{i}" for i in teste["setor"]]
teste

,setor,producao
0,01,13163.401460
1,02,10822.963504
2,03,14697.031250
3,04,11304.365079
4,05,12280.279070
5,06,11943.484848
6,07,12593.157895
7,08,14542.053435
8,09,15315.859375
9,10,15890.161290


In [10]:
mapa = mapa_atual.__geo_interface__

In [18]:
fig = px.choropleth(
    teste,
    geojson=mapa,
    locations="setor",              # ajustar
    featureidkey="properties.SETOR",# ajustar
    color="producao"
)

fig.update_geos(fitbounds="locations", visible=False)
fig.update_layout(width=900,height=900)
fig.show()